# SVM Trial Comparison

Compares validation outputs only. No audio features are loaded or recomputed here.


## 1. Paths


In [1]:
import os
from pathlib import Path

# Purpose: Mounts Google Drive when this notebook is running in Google Colab.
# Why this exists: the shared SVM feature cache, manifests, configs, models,
# figures, and metric outputs may live under /content/drive/MyDrive in Colab.
# A /content/drive path is only reliable after Google Drive has actually been mounted.
# This cell is kept separate and early so the rest of the notebook can resolve paths safely.

COLAB_DRIVE_MOUNT_POINT = Path("/content/drive")
EXPECTED_COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")

try:
    from google.colab import drive

    drive.mount("/content/drive")
    os.environ.setdefault(
        "INTRO_AI_PROJECT_ROOT",
        str(EXPECTED_COLAB_PROJECT_ROOT),
    )
    print("Google Colab detected. Google Drive mounted.")
except Exception as exc:
    print("Google Colab Drive mount skipped. This is expected outside Colab.")
    print("Mount skip reason:", exc)

print("INTRO_AI_PROJECT_ROOT:", os.environ.get("INTRO_AI_PROJECT_ROOT", "not set"))


Mounted at /content/drive
Google Colab detected. Google Drive mounted.
INTRO_AI_PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Education/INM701


In [2]:
import os
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


def resolve_project_root():
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Data").exists() or (candidate / "Datasets").exists() or (candidate / "outputs").exists():
            return candidate
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")

PROJECT_ROOT = resolve_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "svm"
TABLE_DIR = OUTPUT_DIR / "tables"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("TABLE_DIR:", TABLE_DIR)


PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Education/INM701
TABLE_DIR: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/svm/tables


## 2. Load Trial Results


In [3]:
trial_files = {
    "baseline": "svm_baseline_validation.csv",
    "kernel": "svm_kernel_trial.csv",
    "C": "svm_c_trial.csv",
    "gamma": "svm_gamma_trial.csv",
    "class_weight": "svm_class_weight_trial.csv",
}

frames = []
missing = []
for trial_name, filename in trial_files.items():
    path = TABLE_DIR / filename
    if not path.exists():
        missing.append(str(path))
        continue
    frame = pd.read_csv(path)
    frame.insert(0, "trial", trial_name)
    frames.append(frame)

if missing:
    print("Missing trial outputs:")
    for path in missing:
        print(" -", path)

if not frames:
    raise FileNotFoundError("No SVM trial CSV outputs were found. Run notebooks 01-05 first.")

all_trials = pd.concat(frames, ignore_index=True, sort=False)
display(all_trials)


Missing trial outputs:
 - /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/svm/tables/svm_baseline_validation.csv


,trial,variant,split,accuracy,precision,recall,f1,kernel,C,gamma,class_weight
0,kernel,kernel_rbf,validation,0.949537,0.994440,0.932241,0.962337,rbf,1.00,scale,balanced
1,kernel,kernel_linear,validation,0.933574,0.994300,0.909159,0.949825,linear,1.00,scale,balanced
2,kernel,kernel_poly,validation,0.918126,0.996644,0.884587,0.937278,poly,1.00,scale,balanced
3,kernel,kernel_sigmoid,validation,0.727600,0.856392,0.728220,0.787123,sigmoid,1.00,scale,balanced
4,C,C_10,validation,0.968589,0.984871,0.969471,0.977111,rbf,10.00,scale,balanced
5,C,C_100,validation,0.966014,0.977562,0.973194,0.975373,rbf,100.00,scale,balanced
6,C,C_1,validation,0.949537,0.994440,0.932241,0.962337,rbf,1.00,scale,balanced
7,C,C_0.1,validation,0.897528,0.987223,0.862993,0.920938,rbf,0.10,scale,balanced
8,C,C_0.01,validation,0.843460,0.956102,0.810871,0.877518,rbf,0.01,scale,balanced
9,gamma,gamma_auto,validation,0.949537,0.994440,0.932241,0.962337,rbf,1.00,auto,balanced


## 3. Compare Best Validation Result from Each Trial


In [4]:
best_per_trial = (
    all_trials.sort_values("f1", ascending=False)
    .groupby("trial", as_index=False)
    .first()
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

display(best_per_trial)

comparison_path = TABLE_DIR / "svm_individual_trial_comparison.csv"
best_per_trial.to_csv(comparison_path, index=False)
print("Saved comparison:", comparison_path)


,trial,variant,split,accuracy,precision,recall,f1,kernel,C,gamma,class_weight
0,C,C_10,validation,0.968589,0.984871,0.969471,0.977111,rbf,10.0,scale,balanced
1,class_weight,class_weight_none,validation,0.956231,0.981623,0.954579,0.967912,rbf,1.0,scale,balanced
2,gamma,gamma_scale,validation,0.949537,0.994440,0.932241,0.962337,rbf,1.0,scale,balanced
3,kernel,kernel_rbf,validation,0.949537,0.994440,0.932241,0.962337,rbf,1.0,scale,balanced


Saved comparison: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/svm/tables/svm_individual_trial_comparison.csv
